In [ ]:
from plotnine import *
import pandas as pd
import plotly.express as px
import numpy as np

perf_nlmixr = pd.read_csv("outputs/performance_nlmixr.csv")
perf_nlmixr["algo"] = "nlmixr"
perf_pysaem = pd.read_csv("outputs/performance_pysaem.csv")
perf_pysaem_full = perf_pysaem[["time", "nb_patients"]]
perf_pysaem_full["algo"] = "pysaem (train + SAEM)"
perf_pysaem_optim = perf_pysaem[["time_saem", "nb_patients"]].rename(
    columns={"time_saem": "time"}
)
perf_pysaem_optim["algo"] = "pysaem (SAEM only)"

full_df = pd.concat([perf_nlmixr, perf_pysaem_full, perf_pysaem_optim])

In [ ]:
display(full_df)

In [ ]:
fig = px.scatter(
    full_df,
    y="time",
    x="nb_patients",
    color="algo",
    log_x=True,
    log_y=True,
    title="Run time for SAEM on TMDD model<br>(100/100 iterations, 1 chain, 5x) depending on observed data set size",
    labels={
        "time": "Runtime (s)",
        "nb_patients": "Number of patients",
        "algo": "Implementation",
    },
    template="ggplot2",
    trendline="ols",
    trendline_options=dict(log_x=True, log_y=True),
    opacity=0.5,
)
fig.update_traces(marker=dict(size=10))
fig.update_layout(
    boxgap=0.0,
    boxgroupgap=0.0,
    width=800,
    height=400,
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.3),
    margin=dict(t=50, l=30, r=30, b=10),
)
fig.show()

In [ ]:
def process_ebe_data(name):
    ebe_pysaem = pd.read_csv("outputs/ebe_pysaem_" + name + ".csv")
    ebe_pysaem["algo"] = "pysaem"
    ebe_pysaem["setting"] = name
    ebe_nlmixr = pd.read_csv("outputs/ebe_nlmixr_" + name + ".csv")
    ebe_nlmixr["algo"] = "nlmixr"
    ebe_nlmixr["setting"] = name
    ebe = pd.concat([ebe_pysaem, ebe_nlmixr])
    descriptors = ["k_eL", "R0", "Vc"]

    ebe = ebe.melt(
        id_vars=["id", "algo", "setting"], value_vars=descriptors, var_name="descriptor"
    ).reset_index()

    return ebe

In [ ]:
ebe_obs_1d = process_ebe_data("1dose")
ebe_obs_2d = process_ebe_data("2dose")
ebe_obs = pd.concat([ebe_obs_1d, ebe_obs_2d])

ebe_true_2d = (
    pd.read_csv("data/obs_data_100.csv")[
        ["id", "protocol_arm", "true_k_eL", "true_R0", "true_Vc"]
    ]
    .drop_duplicates()
    .rename(columns={"true_k_eL": "k_eL", "true_R0": "R0", "true_Vc": "Vc"})
    .melt(
        id_vars=["id", "protocol_arm"],
        value_vars=["k_eL", "R0", "Vc"],
        var_name="descriptor",
        value_name="true_value",
    )
)
ebe_true_1d = ebe_true_2d.loc[ebe_true_2d["protocol_arm"] == "arm_1"]
ebe_true_1d = ebe_true_1d.drop(columns=["protocol_arm"])
ebe_true_2d = ebe_true_2d.drop(columns=["protocol_arm"])
ebe_true_1d["setting"] = "1dose"
ebe_true_2d["setting"] = "2dose"
ebe_true = pd.concat([ebe_true_1d, ebe_true_2d])

comp_df = pd.concat(
    [ebe_obs, ebe_true.rename(columns={"true_value": "value"}).assign(algo="true")]
)
display(comp_df.head())

In [ ]:
(
    ggplot(comp_df, aes(x="value", fill="algo", color="algo"))
    + geom_density(alpha=0.3)
    + facet_grid(cols="descriptor", rows="setting", scales="free")
    + scale_x_continuous(trans="log10")
    + theme_minimal()
    + theme(figure_size=(10, 4))
)